In [116]:
import optuna

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

In [117]:
study_name = "hpo_multi_qrs_aligned"
sqlite_db_path = "results/hpo_multi_qrs_aligned/BiXLSTM1new/db.sqlite3"


study = optuna.create_study(study_name=study_name, storage=f'sqlite:///{sqlite_db_path}', load_if_exists=True)

[I 2025-05-15 00:07:08,238] Using an existing study with name 'hpo_multi_qrs_aligned' instead of creating a new one.


In [118]:
print(study.trials[0].user_attrs)
print((study.trials[0].params))

{'block_config': '6_smsmmm', 'num_blocks_slstm_at': '6_[0,2]'}
{'num_blocks_slstm_at': 69, 'dropout': 0.5, 'embedding_dim': 24, 'conv1d_kernel_size': 21, 'num_heads': 4}


In [119]:
config_dict = {
        t.params["num_blocks_slstm_at"]: {
            "config_str": t.user_attrs["num_blocks_slstm_at"],
            "num_blocks": t.user_attrs["num_blocks_slstm_at"].split("_")[0],
            "slstm_at": t.user_attrs["num_blocks_slstm_at"].split("_")[1],
        } for t in study.trials
    }

def get_config(id):
    config_dict[id]["num_blocks"], config_dict[id]["slstm_at"]
def get_config_str(id):
    config_dict[id]["config_str"]

In [120]:
plot_optimization_history(study, target=lambda t: t.values[0], target_name="macro f1")

In [121]:
plot_optimization_history(study, target=lambda t: t.values[1], target_name="number of parameters")

In [122]:
plot_intermediate_values(study)

[W 2025-05-15 00:07:08,283] You need to set up the pruning feature to utilize `plot_intermediate_values()`


In [123]:
plot_slice(study, target = lambda t: t.values[0], target_name="macro f1")

In [124]:
plot_slice(study, target = lambda t: t.values[1], target_name="number of parameters")

In [125]:
fig = optuna.visualization.plot_pareto_front(study, targets=lambda t: (t.values[1], t.values[0]), target_names=["# Parameters", "macro f1"])
fig.show()

In [126]:
fig = optuna.visualization.plot_contour(study, target = lambda t: t.values[0], target_name = ["macro f1"])
fig.show()

[W 2025-05-15 00:07:08,344] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,345] Param num_heads unique value length is less than 2.
[W 2025-05-15 00:07:08,346] Param num_heads unique value length is less than 2.


ValueError: 
    Invalid value of type 'builtins.list' received for the 'title' property of contour.colorbar
        Received value: ['macro f1']

    The 'title' property is an instance of Title
    that may be specified as:
      - An instance of :class:`plotly.graph_objs.contour.colorbar.Title`
      - A dict of string/value properties that will be passed
        to the Title constructor

        Supported dict properties:
            
            font
                Sets this color bar's title font.
            side
                Determines the location of color bar's title
                with respect to the color bar. Defaults to
                "top" when `orientation` if "v" and  defaults
                to "right" when `orientation` if "h".
            text
                Sets the title of the color bar.


In [ ]:
plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study, target=lambda t: t.values[0], target_name="macro f1")

In [ ]:
plot_parallel_coordinate(study, target=lambda t: t.values[1], target_name="number of parameters")

In [132]:
import plotly.express as px
import pandas as pd
from collections import OrderedDict
import random

dd = OrderedDict({})

#print(px.data.iris())


for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        dd[trial.number] = OrderedDict({
            "num_blocks": int(trial.user_attrs["num_blocks_slstm_at"].split("_")[0]),
            "slstm_at": trial.user_attrs["num_blocks_slstm_at"].split("_")[1],
            "config_id": trial.params["num_blocks_slstm_at"],
            "dropout": trial.params["dropout"],
            "embedding_dim": trial.params["embedding_dim"],
            "conv1d_kernel_size": trial.params["conv1d_kernel_size"],
            "macro_f1": trial.values[0],
            "total_params": trial.values[1],
            #"macro_f1": 0.993,
            #"total_params": 43556,
            "num_heads": trial.params["num_heads"],
        })


column_names = list(dd[0].keys())

d = {
    i: [dd[i][k] for k in column_names] for i in range(len(dd))
}

df = pd.DataFrame.from_dict(d, orient='index', columns=column_names)

print((df))

#color = "total_params"
color = "macro_f1"

#color_midpoint = 0.5 if color == "macro_f1" else max(df["total_params"])/2
#color_midpoint = max(df["macro_f1"])/2 if color == "macro_f1" else max(df["total_params"])/2

range_color = [min(df["macro_f1"]), max(df["macro_f1"])] if color == "macro_f1" else [min(df["total_params"]), max(df["total_params"])]

fig = px.parallel_coordinates(
    df, 
    color=color, 
    labels={
        "num_blocks": "xlstm blocks", 
        #"slstm_at": "slstm at", 
        "dropout": "Dropout",
        "conv1d_kernel_size": "Conv1D Kernel Size",
        "embedding_dim": "Embedding dimension", 
        "macro_f1": "Macro F1 Score",
        "total_params": "Number of Parameters",
        "num_heads": "Number of Heads", 
    },
    dimensions=[
        "num_blocks", 
        #"slstm_at", 
        "dropout", 
        "embedding_dim", 
        "conv1d_kernel_size", 
        "macro_f1",
        "total_params",
        "num_heads", 
    ],
    color_continuous_scale=px.colors.diverging.Tealrose,
    color_continuous_midpoint=(range_color[1]-range_color[0])/2,
    range_color=range_color,)

# fig = px.parallel_coordinates(
#     df, 
#     color="species_id", 
#     labels={"species_id": "Species","sepal_width": "Sepal Width", "sepal_length": "Sepal Length", "petal_width": "Petal Width", "petal_length": "Petal Length", },
#     color_continuous_scale=px.colors.diverging.Tealrose,
#     color_continuous_midpoint=2)

# Hide the color scale that is useless in this case
fig.update_layout(coloraxis_showscale=False)

# Show the plot
fig.show()

   num_blocks slstm_at  config_id  dropout  embedding_dim  conv1d_kernel_size  \
0           6    [0,2]         69     0.50             24                  21   
1           4  [0,1,3]         26     0.25             40                  11   
2           2    [0,1]          5     0.10             16                  11   

   macro_f1  total_params  num_heads  
0  0.906737       45564.0          4  
1  0.907529       56404.0          4  
2  0.889891        7956.0          4  


In [143]:
import plotly.graph_objects as go


fig = go.Figure(data=
    go.Parcoords(
        line = dict(color = df['macro_f1'],
                   colorscale = [[0,'purple'],[0.5,'lightseagreen'],[1,'gold']],
                   showscale = True,
                   cmin = df['macro_f1'].min(),
                   cmax = df['macro_f1'].max(),
        ),
     #    line = dict(color = df['total_params'],
     #               colorscale = [[df['total_params'].min(),'purple'],[(df['total_params'].max()-df['total_params'].min())/2,'lightseagreen'],[df['total_params'].max(),'gold']],
     #               showscale = True,
     #               cmin = df['total_params'].min(),
     #               cmax = df['total_params'].max(),
     #    ),
        dimensions = list([
            dict(range = [df['num_blocks'].min(),df['num_blocks'].max()],
                 label = "num blocks", values = df['num_blocks']),
            dict(range = [df['config_id'].min(),df['config_id'].max()],
                 tickvals = sorted(df['config_id']),
                 label = 'slstm at', values = df['config_id'],
                 ticktext = [config_dict[id]["slstm_at"] for id in sorted(df['config_id'])]),
            dict(range = [df['dropout'].min(),df['dropout'].max()],
                 label = "Dropout", values = df['dropout']),
            dict(range = [df['embedding_dim'].min(),df['embedding_dim'].max()],
                 label = "embedding_dim", values = df['embedding_dim']),
            dict(range = [df['conv1d_kernel_size'].min(),df['conv1d_kernel_size'].max()],
                 label = "conv1d_kernel_size", values = df['conv1d_kernel_size']),
            dict(range = [df['macro_f1'].min(),df['macro_f1'].max()],
                 label = "macro_f1", values = df['macro_f1']),
            dict(range = [df['total_params'].min(),df['total_params'].max()],
                 label = "total_params", values = df['total_params']),
            dict(range = [df['num_heads'].min(),df['num_heads'].max()],
                 label = "num_heads", values = df['num_heads']),
            ]
          )
    )
)
fig.show()


In [ ]:
plot_contour(study, target=lambda t: t.values[0], target_name="macro f1")

[W 2025-05-14 23:57:39,854] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,854] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,854] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,854] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,855] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,855] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,855] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:39,855] Param num_heads unique value length is less than 2.


In [ ]:
plot_contour(study, target=lambda t: t.values[1], target_name="number of parameters")

[W 2025-05-14 23:57:56,198] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,199] Param num_heads unique value length is less than 2.
[W 2025-05-14 23:57:56,200] Param num_heads unique value length is less than 2.


In [ ]:
plot_rank(study, target = lambda t: t.values[0], target_name="macro f1")


In [ ]:
plot_rank(study, target = lambda t: t.values[1], target_name="number of parameters")

In [ ]:
plot_timeline(study)

In [130]:
print(len(study.trials))
for t in study.trials:
    print(f"state: {t.state} number: {t.number}")

4
state: 1 number: 0
state: 1 number: 1
state: 1 number: 2
state: 0 number: 3
